# HUẤN LUYỆN TabNet TRÊN DỮ LIỆU ĐÃ SMOTE (mục 4.3.1)

Notebook này là bản sao của `TABNET.ipynb`, chỉ khác ở chỗ:

1. **Tập train** đọc từ `X_train_smote.csv` / `y_train_smote.csv` (do `pre_smote.ipynb` tạo ra), thay vì `X_train.csv` / `y_train.csv` gốc.
2. **Tập validation và test giữ nguyên bản gốc** (`X_val.csv`, `X_test.csv`, `y_val.csv`, `y_test.csv`) — đúng như luận văn mô tả: *"TabNet được huấn luyện lại trên tập train đã SMOTE, giữ nguyên tập validation và test gốc để đảm bảo phép so sánh công bằng"*.
3. **Tham số TabNet giữ y hệt** bản gốc (`n_d=64, n_a=64, n_steps=5, ...`) để phép so sánh gốc vs SMOTE là công bằng — không đổi để "ăn gian" kết quả.
4. Toàn bộ file output (model, ảnh, JSON) được đặt **tên khác** bản gốc (hậu tố `_smote`) để **không ghi đè** lên kết quả TabNet gốc đã có.

**Chạy sau khi đã có:** `X_train_smote.csv`, `y_train_smote.csv` (từ `pre_smote.ipynb`) và `X_val.csv`, `X_test.csv`, `y_val.csv`, `y_test.csv`, `label_encoder.pkl` (đã có sẵn từ `pre.ipynb`).

**Output chính:** `results_smote_real.json` — đúng schema với `results.json` gốc (`accuracy`, `macro_avg`, `weighted_avg`, `per_class_metrics`, `confusion_matrix`), để dùng trực tiếp trong tab "SMOTE" của dashboard `DEMO.py` (Trang 7).


In [ ]:
# !pip install pytorch-tabnet


In [ ]:
import pandas as pd
import numpy as np
import time
import json
import pickle
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Accuracy, BalancedAccuracy
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ==============================================================================
# 1. TẢI DỮ LIỆU — train lấy bản SMOTE, val/test giữ nguyên bản gốc
# ==============================================================================
data_dir = "../data/processed/"

X_train = pd.read_csv(data_dir + "X_train_smote.csv")
X_val   = pd.read_csv(data_dir + "X_val.csv")
X_test  = pd.read_csv(data_dir + "X_test.csv")

y_train = pd.read_csv(data_dir + "y_train_smote.csv").values.ravel()
y_val   = pd.read_csv(data_dir + "y_val.csv").values.ravel()
y_test  = pd.read_csv(data_dir + "y_test.csv").values.ravel()

with open(data_dir + "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

label_names = label_encoder.classes_

print(f"Train set (SMOTE): {X_train.shape}, Labels: {y_train.shape}")
print(f"Validation set (gốc): {X_val.shape}, Labels: {y_val.shape}")
print(f"Test set (gốc): {X_test.shape}, Labels: {y_test.shape}")
print(f"Số lớp: {len(np.unique(y_train))}")
print(f"Số features: {X_train.shape[1]}")

print("\nNhãn các lớp:")
for idx, name in enumerate(label_names):
    print(f"  {idx}: {name}")

print("\nPhân bố lớp trên tập train SMOTE (đối chiếu Bảng 4.2):")
unique, counts = np.unique(y_train, return_counts=True)
for idx, cnt in zip(unique, counts):
    print(f"  {label_names[idx]:<25} {cnt:>10,d}")


In [ ]:
# ==============================================================================
# 2. THAM SỐ TabNet — GIỮ NGUYÊN như TABNET.ipynb để so sánh công bằng
# ==============================================================================
tabnet_params = {
    "n_d": 64,
    "n_a": 64,
    "n_steps": 5,
    "gamma": 1.5,
    "n_independent": 2,
    "n_shared": 2,
    "lambda_sparse": 1e-4,
    "momentum": 0.3,
    "clip_value": 2.0,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": dict(lr=2e-2),
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {
        "step_size": 10,
        "gamma": 0.9
    },
    "mask_type": "entmax",
    "verbose": 1,
    "seed": 42
}

print("Tham số TabNet (giống hệt bản gốc):")
for key, value in tabnet_params.items():
    if not callable(value) and not isinstance(value, dict):
        print(f"  {key}: {value}")

model = TabNetClassifier(**tabnet_params)


In [ ]:
# ==============================================================================
# 3. HUẤN LUYỆN
# ==============================================================================
timing_smote = {}
t0 = time.perf_counter()

model.fit(
    X_train=X_train.values,
    y_train=y_train,
    eval_set=[(X_train.values, y_train), (X_val.values, y_val)],
    eval_name=["train", "valid"],
    eval_metric=["accuracy", "balanced_accuracy"],
    max_epochs=100,
    patience=15,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

timing_smote['tabnet_training_sec'] = time.perf_counter() - t0
n_epochs_actual = len(model.history.history['loss'])
timing_smote['tabnet_epochs_actual'] = n_epochs_actual
timing_smote['tabnet_sec_per_epoch'] = timing_smote['tabnet_training_sec'] / n_epochs_actual

print(f"Thời gian huấn luyện TabNet+SMOTE: {timing_smote['tabnet_training_sec']:.1f}s "
      f"({n_epochs_actual} epochs, {timing_smote['tabnet_sec_per_epoch']:.2f}s/epoch)")

# Lưu riêng, KHÔNG ghi đè timing_training.json của bản gốc
with open(data_dir + 'timing_training_smote.json', 'w', encoding='utf-8') as f:
    json.dump(timing_smote, f, indent=4)
print(f"✓ Đã lưu: {data_dir}timing_training_smote.json")


In [ ]:
# ==============================================================================
# 4. LƯU MODEL + HISTORY (tên riêng, không đè bản gốc)
# ==============================================================================
history_to_save = {k: [float(x) for x in v] for k, v in model.history.history.items()}
with open(data_dir + 'training_history_smote.json', 'w') as f:
    json.dump(history_to_save, f)
print("Đã lưu history SMOTE. Keys:", list(history_to_save.keys()))

model_path = data_dir + "tabnet_iec104_model_smote"
model.save_model(model_path)
print(f"Đã lưu model tại: {model_path}.zip")


In [ ]:
# ==============================================================================
# 5. DỰ ĐOÁN TRÊN TẬP TEST GỐC (không đổi)
# ==============================================================================
y_pred = model.predict(X_test.values)
y_pred_proba = model.predict_proba(X_test.values)


In [ ]:
# ==============================================================================
# 6. TÍNH CÁC CHỈ SỐ ĐÁNH GIÁ — giống hệt cách tính trong TABNET.ipynb
# ==============================================================================
acc = accuracy_score(y_test, y_pred)
print(f"\n1. ACCURACY: {acc:.4f} ({acc*100:.2f}%)")

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, labels=range(len(label_names))
)

try:
    from sklearn.preprocessing import label_binarize
    y_test_bin = label_binarize(y_test, classes=range(len(label_names)))

    auc_scores = []
    for i in range(len(label_names)):
        try:
            auc = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
            auc_scores.append(auc)
        except Exception:
            auc_scores.append(np.nan)

    macro_auc = np.nanmean(auc_scores)
    weighted_auc = np.average(
        [auc_scores[i] for i in range(len(auc_scores)) if not np.isnan(auc_scores[i])],
        weights=[support[i] for i in range(len(support)) if not np.isnan(auc_scores[i])]
    )
except Exception as e:
    print(f"\nCảnh báo: Không thể tính AUC - {e}")
    auc_scores = [np.nan] * len(label_names)
    macro_auc = np.nan
    weighted_auc = np.nan

print("\n2. CHI TIẾT THEO TỪNG LỚP:")
print("-" * 100)
print(f"{'Class':<25} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'AUC':<12} {'Support':<10}")
print("-" * 100)
for idx, name in enumerate(label_names):
    auc_str = f"{auc_scores[idx]:.4f}" if not np.isnan(auc_scores[idx]) else "N/A"
    print(f"{name:<25} {precision[idx]:<12.4f} {recall[idx]:<12.4f} "
          f"{f1[idx]:<12.4f} {auc_str:<12} {support[idx]:<10}")
print("-" * 100)

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print(f"\n3. TRUNG BÌNH:")
print("-" * 100)
print(f"{'Metric':<25} {'Macro Avg':<15} {'Weighted Avg':<15}")
print("-" * 100)
print(f"{'Precision':<25} {macro_p:<15.4f} {weighted_p:<15.4f}")
print(f"{'Recall':<25} {macro_r:<15.4f} {weighted_r:<15.4f}")
print(f"{'F1-Score':<25} {macro_f1:<15.4f} {weighted_f1:<15.4f}")
auc_macro_str = f"{macro_auc:.4f}" if not np.isnan(macro_auc) else "N/A"
auc_weighted_str = f"{weighted_auc:.4f}" if not np.isnan(weighted_auc) else "N/A"
print(f"{'AUC':<25} {auc_macro_str:<15} {auc_weighted_str:<15}")
print("-" * 100)


In [ ]:
# ==============================================================================
# 7. CONFUSION MATRIX (tên file riêng)
# ==============================================================================
cm = confusion_matrix(y_test, y_pred)
cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

annot = np.empty_like(cm, dtype=object)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        count = cm[i, j]
        pct = cm_percent[i, j]
        annot[i, j] = '' if count == 0 else f'{count}\n({pct:.1f}%)'

plt.figure(figsize=(13, 10))
from matplotlib.colors import LogNorm
mask_zero = (cm == 0)

ax = sns.heatmap(
    cm, annot=annot, fmt='', cmap='Blues',
    norm=LogNorm(vmin=1, vmax=cm.max()),
    xticklabels=label_names, yticklabels=label_names,
    linewidths=0.5, linecolor='gray',
    cbar_kws={'label': 'Count (log scale)'},
    mask=mask_zero
)
for i in range(len(label_names)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='red', lw=2))

plt.title('Confusion Matrix — TabNet + SMOTE on SANDI-2024/IEC 104', fontsize=14, pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(data_dir + 'confusion_matrix_smote.png', dpi=300, bbox_inches='tight')
print("✓ Đã lưu confusion_matrix_smote.png")
plt.close()


In [ ]:
# ==============================================================================
# 8. FEATURE IMPORTANCE + LEARNING CURVES + ROC (tên file riêng)
# ==============================================================================
feature_importances = model.feature_importances_
feature_names = X_train.columns
indices = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(12, 8))
top_n = 20
top_indices = indices[:top_n]
plt.barh(range(top_n), feature_importances[top_indices])
plt.yticks(range(top_n), [feature_names[i] for i in top_indices])
plt.xlabel('Importance Score', fontsize=12)
plt.title(f'Top {top_n} Feature Importances - TabNet + SMOTE', fontsize=14, pad=20)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(data_dir + 'feature_importance_smote.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Đã lưu: {data_dir}feature_importance_smote.png")

try:
    history_dict = model.history.history
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history_dict['train_balanced_accuracy'], label='Train Balanced Acc', linewidth=2, color='steelblue')
    plt.plot(history_dict['valid_balanced_accuracy'], label='Validation Balanced Acc', linewidth=2,
             linestyle='--', color='orange', marker='o', markersize=3)
    plt.title('Training and Validation Balanced Accuracy (SMOTE)', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Balanced Accuracy', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    if 'train_accuracy' in history_dict:
        plt.plot(history_dict['train_accuracy'], label='Train Accuracy', linewidth=2, color='steelblue', alpha=0.9)
    if 'valid_accuracy' in history_dict:
        plt.plot(history_dict['valid_accuracy'], label='Validation Accuracy', linewidth=2,
                 linestyle='--', color='orange', alpha=0.9, marker='o', markersize=3)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Training and Validation Accuracy (SMOTE)', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(data_dir + 'learning_curves_smote.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Đã lưu: {data_dir}learning_curves_smote.png")
except Exception as e:
    print(f"Lỗi khi vẽ learning curves: {e}")

from sklearn.metrics import roc_curve, auc as auc_fn
classes = sorted(np.unique(y_test))
y_test_bin2 = label_binarize(y_test, classes=classes)

plt.figure(figsize=(10, 8))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin2[:, i], y_pred_proba[:, i])
    roc_auc = auc_fn(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=1.5, label=f'{label_names[i]} (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Đường cong ROC đa lớp — TabNet + SMOTE trên SANDI-2024/IEC 104', fontsize=13)
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(data_dir + 'roc_curve_smote.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Đã lưu roc_curve_smote.png")


In [ ]:
# ==============================================================================
# 9. LƯU KẾT QUẢ — ĐÚNG SCHEMA với results.json gốc, để DEMO.py đọc được ở
#    tab "SMOTE" (Trang 7). Tên file: results_smote_real.json
#    (khớp với FILES['results_smote_json'] đã cấu hình trong DEMO.py)
# ==============================================================================
results_smote = {
    'accuracy': float(acc),
    'macro_avg': {
        'precision': float(macro_p),
        'recall': float(macro_r),
        'f1_score': float(macro_f1),
        'auc': float(macro_auc) if not np.isnan(macro_auc) else None
    },
    'weighted_avg': {
        'precision': float(weighted_p),
        'recall': float(weighted_r),
        'f1_score': float(weighted_f1),
        'auc': float(weighted_auc) if not np.isnan(weighted_auc) else None
    },
    'per_class_metrics': {
        label_names[i]: {
            'precision': float(precision[i]),
            'recall': float(recall[i]),
            'f1_score': float(f1[i]),
            'auc': float(auc_scores[i]) if not np.isnan(auc_scores[i]) else None,
            'support': int(support[i])
        }
        for i in range(len(label_names))
    },
    'confusion_matrix': cm.tolist()
}

with open(data_dir + 'results_smote_real.json', 'w', encoding='utf-8') as f:
    json.dump(results_smote, f, indent=4)

print(f"✓ Đã lưu kết quả tại: {data_dir}results_smote_real.json")


In [ ]:
# ==============================================================================
# 10. TÓM TẮT — đối chiếu nhanh với số liệu TabNet gốc (Bảng 4.1) nếu có sẵn
# ==============================================================================
print("=" * 65)
print("  KẾT QUẢ TabNet + SMOTE trên tập test GỐC")
print("=" * 65)
print(f"  Accuracy:          {acc*100:.2f}%")
print(f"  Macro Precision:   {macro_p:.4f}")
print(f"  Macro Recall:      {macro_r:.4f}")
print(f"  Macro F1-Score:    {macro_f1:.4f}")
print(f"  Macro AUC:         {auc_macro_str}")

try:
    with open(data_dir + 'results_smote.json', 'r') as f:
        # LƯU Ý: results_smote.json (đã có sẵn) thực chất là kết quả TabNet
        # GỐC (không SMOTE) do notebook TABNET.ipynb đặt nhầm tên khi lưu.
        base = json.load(f)
    base_macro = base.get('macro_avg', {})
    print("\n  So với TabNet GỐC (đọc từ results_smote.json, xem lưu ý ở DEMO.py):")
    print(f"    Macro F1 gốc:  {base_macro.get('f1_score', float('nan')):.4f}")
    print(f"    Macro F1 SMOTE:{macro_f1:.4f}  (chênh lệch: {macro_f1 - base_macro.get('f1_score', 0):+.4f})")
except FileNotFoundError:
    print("\n  (Không tìm thấy file kết quả TabNet gốc để so sánh trực tiếp.)")

print("\nCác file đã lưu (đều có hậu tố _smote hoặc tên riêng, không đè bản gốc):")
for fname in [
    'timing_training_smote.json', 'training_history_smote.json',
    'tabnet_iec104_model_smote.zip', 'confusion_matrix_smote.png',
    'feature_importance_smote.png', 'learning_curves_smote.png',
    'roc_curve_smote.png', 'results_smote_real.json',
]:
    print(f"  {fname}")
print("=" * 65)
